# Impacto de los Ingresos y las Relaciones Interpersonales en la Felicidad
## Investigación mediante un Modelo Basado en Agentes (ABM) Calibrado con Microdatos del CIS (Versión VSCODE + Jupyter Notebook)

**Autor:** Julian Carrion Tovar y Fernando Jose Gracia Choin

---

## Introducción
Este cuaderno presenta un flujo de trabajo completo para investigar la relación entre el nivel de ingresos y la felicidad autopercibida, mediada por las relaciones interpersonales y el uso de redes sociales. 

El proyecto se divide en tres fases críticas:
1. **Procesamiento de Datos:** Filtrado de microdatos reales del estudio 3145 del CIS.
2. **Fundamentación Matemática:** Derivación de perfiles de felicidad y sociabilidad basados en la renta.
3. **Simulación Social:** Dinámicas de contagio emocional y movilidad en un entorno artificial.

### Sección 0: Preparación del Entorno
En esta celda instalamos las librerías necesarias (`mesa` para la simulación, `pandas` para datos, etc.) y configuramos las rutas de carpetas del proyecto para que el cuaderno funcione de forma self-contained.

In [ ]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# Función para auto-instalar dependencias si no existen
def install_packages():
    packages = ["mesa", "pandas", "numpy", "matplotlib", "ipywidgets", "openpyxl"]
    print(f"Instalando dependencias: {', '.join(packages)}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)

try:
    import mesa
    import matplotlib.pyplot as plt
    import ipywidgets as widgets
except ImportError:
    install_packages()
    print("\nơInstalación completada! REINICIA EL KERNEL y vuelve a ejecutar esta celda.")

# Configuración de la estructura de archivos relative al notebook
BASE_DIR = ".."
CLEAN_DATA_PATH = os.path.join(BASE_DIR, "documentos_datos", "clean_data")
DATA_PATH = os.path.join(BASE_DIR, "documentos_datos", "data")

# Crear carpeta de salida si no existe
if not os.path.exists(CLEAN_DATA_PATH): 
    os.makedirs(CLEAN_DATA_PATH)
    print(f"Directorio creado: {CLEAN_DATA_PATH}")

## Sección 1: Filtrado y Limpieza de Microdatos
Utilizamos el estudio **3145 del CIS** ("Redes Sociales y Felicidad"). El objetivo es extraer perfiles de usuarios de diferentes redes sociales (Facebook, Instagram y X) que cumplan criterios de validez estadística.

**Criterios de limpieza:**
- Usuarios activos en la red social correspondiente.
- Personas con felicidad autopercibida definida.
- Eliminación de valores nulos o errores de encuesta (valores >= 90).

In [ ]:
input_file = os.path.join(DATA_PATH, "3145_data.xlsx")
print("Cargando microdatos del estudio 3145...")

try:
    df_full = pd.read_excel(input_file)
    
    def filtrar_red(social_key, col_name):
        """Filtra y guarda datos específicos por cada red social"""
        out_path = os.path.join(CLEAN_DATA_PATH, f"3145_data_clean_{social_key}.xlsx")
        
        # Filtrado según respuestas válidas
        f = df_full[(
            (df_full[col_name] == 1) & # Usuario de la red
            (df_full["P60A"] == 1) &   # Tiene perfil
            (df_full["P65"] < 90) &    # Renta válida
            (df_full["P69"] < 90)      # Felicidad válida
        )].copy()
        
        # Re-mapeo manual de escalas para normalización
        f["P69"] = f["P69"].replace({0:0, 1:0, 2:1, 3:1, 4:2, 5:2, 6:3, 7:3, 8:4, 9:4, 10:5})
        
        f.to_excel(out_path, index=False)
        print(f" -> Datos de {social_key} guardados: {len(f)} registros")

    # Procesamos las tres redes principales
    filtrar_red("FB", "P21A01") # Facebook
    filtrar_red("IG", "P21A05") # Instagram
    filtrar_red("X", "P21A02")  # X (Twitter)
    
    print("\n¡Procesamiento finalizado con éxito!")
except Exception as e:
    print(f"Error al cargar los datos: {e}. Asegúrate de que el archivo Excel está en la subcarpeta 'documentos_datos/data'.")

## Sección 2: Generación de Perfiles Matemáticos
Para la simulación, convertimos los datos brutos del CIS en parámetros de comportamiento del agente. 

Aplicamos un coeficiente $\alpha = 0.5$ para equilibrar el peso de los ingresos en la felicidad inicial, siguiendo la fórmula:
$$Happiness = \frac{(Income^\alpha \cdot 8^{(1-\alpha)}) \cdot 5}{11}$$

También definimos la *sociabilidad* como una función de la propia felicidad, permitiendo que agentes más satisfechos tengan más probabilidad de interactuar.

In [ ]:
def generar_perfiles(alpha, dataset_name):
    """Genera un dataset calibrado con parámetros de simulación"""
    data_file = os.path.join(CLEAN_DATA_PATH, f"3145_data_clean_{dataset_name}.xlsx")
    if not os.path.exists(data_file): return
    
    df = pd.read_excel(data_file)
    
    # Cálculo de felicidad base
    h = [round(((i ** alpha * 8.0 ** (1 - alpha)) * 5) / 11, 2) for i in df["P65"]]
    # Cálculo de sociabilidad derivada
    s = [round(val * (1 + (val ** (1 - alpha))), 2) for val in h]
    
    # Exportación del modelo final por red social
    output_df = pd.DataFrame({"H": h, "S": s, "I": df["P65"]})
    output_df.to_excel(os.path.join(CLEAN_DATA_PATH, f"model_{dataset_name}.xlsx"), index=False)
    print(f"Modelo calibrado para {dataset_name} generado.")

for net in ["FB", "IG", "X"]: 
    generar_perfiles(0.5, net)
print("\nFase de modelado matemático completada.")

## Sección 3: El Modelo Basado en Agentes (ABM)
Aquí definimos la lógica interna de la sociedad artificial. 

### Dinámicas del Agente:
1. **Impacto del SMI:** Si se aplica un Salario Mínimo Interprofesional, los agentes que ganan menos ven incrementada su felicidad base.
2. **Movilidad:** Los agentes más infelices tienden a moverse por el mapa buscando nuevas conexiones sociales.
3. **Contagio Social:** En cada paso, la felicidad del agente se ajusta ligeramente hacia la media de sus vecinos (emulando la presión social o el impacto de las relaciones interpersonales).

In [ ]:
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector

# Comprobación de versión de Mesa para garantizar compatibilidad híbrida
HAS_SCHEDULER = False
try:
    from mesa.time import RandomActivation
    HAS_SCHEDULER = True
except ImportError: pass

# Conversión de códigos CIS a valores económicos reales aproximados
MAPA_EQUIV = {1:0, 2:300, 3:450, 4:750, 5:1050, 6:1500, 7:2100, 8:2700, 9:3750, 10:5250, 11:7000}

class DynamicSocialAgent(Agent):
    """Representación de un ciudadano con renta y felicidad propia"""
    def __init__(self, unique_id, model, income_code, happiness_val, sociability_val):
        # Inicialización compatible con Mesa 2.x y 3.x
        try: super().__init__(unique_id, model)
        except TypeError: 
            super().__init__(model)
            self.unique_id = unique_id
            
        self.income_code = int(income_code)
        self.income_val = MAPA_EQUIV.get(self.income_code, 300)
        self.happiness = float(happiness_val)
        self.base_h = self.happiness # Felicidad base heredada del CIS
        self.sociability = float(sociability_val)

    def step(self):
        # REGLA 1: Impacto del SMI
        eff_inc = max(self.income_val, self.model.min_wage)
        curr_h = self.base_h + (((eff_inc - self.income_val)/1000)*0.8 if eff_inc > self.income_val else 0)
        
        # REGLA 2: Movilidad Social (buscan mejores entornos si están mal)
        if self.happiness < 2.0 and self.random.random() < 0.2: 
            new_pos = self.random.choice(self.model.grid.get_neighborhood(self.pos, moore=True))
            self.model.grid.move_agent(self, new_pos)
        
        # REGLA 3: Contagio Emocional con vecinos
        nb = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        if nb: 
            self.happiness += (np.mean([n.happiness for n in nb]) - self.happiness) * 0.05
        
        # Retorno a la estabilidad individual gradual
        self.happiness += (curr_h - self.happiness) * 0.1
        self.happiness = max(0, min(5, self.happiness))

class SocialEvolutionModel(Model):
    """Entorno macro de la simulación social"""
    def __init__(self, N, width, height, filename, min_wage=1000):
        super().__init__()
        self.grid = MultiGrid(width, height, True)
        self.min_wage = min_wage
        
        if HAS_SCHEDULER: self.schedule = RandomActivation(self)
        
        # Cargar datos calibrados de la red social elegida
        df = pd.read_excel(os.path.join(CLEAN_DATA_PATH, filename))
        
        for i in range(N):
            idx = i % len(df)
            a = DynamicSocialAgent(i, self, df.iloc[idx,2], df.iloc[idx,0], df.iloc[idx,1])
            if HAS_SCHEDULER: self.schedule.add(a)
            self.grid.place_agent(a, (self.random.randrange(width), self.random.randrange(height)))
            
        # Iniciamos el recolector de datos para análisis
        self.datacollector = DataCollector({"H": lambda m: np.mean([a.happiness for a in (m.agents if hasattr(m, "agents") else m.schedule.agents)])})
        self.steps_count = 0
        self.datacollector.collect(self)

    def step(self):
        # Ejecución según versión de Mesa
        if HAS_SCHEDULER: self.schedule.step()
        else: self.agents.shuffle().do("step")
        
        self.steps_count += 1
        self.datacollector.collect(self)

## Sección 4: Dashboard Interactivo de Visualización
Esta es la pieza central para experimentar con el modelo. Permite ajustar el **SMI** en tiempo real y observar cómo evoluciona el bienestar según el nivel de renta.

**Cómo usarlo:**
1. Selecciona una **Red Social** (la población cambiará).
2. Ajusta el **SMI (€)** (mueves el listón de ingresos mínimos).
3. Pulsa **Resetear** para crear el mundo artificial.
4. Pulsa **Siguiente Paso** para ver la evolución temporal.

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# Contenedores dedicados para evitar duplicación de salida
dashboard_out = widgets.Output()
stats_out = widgets.Output()

def get_agent_color(h):
    """Mapeo original de 5 niveles de felicidad"""
    if h <= 1.5: return "#d63031" # Rojo Oscuro
    if h <= 2.5: return "#ff7675" # Rojo
    if h <= 3.5: return "#fdcb6e" # Dorado
    if h <= 4.5: return "#55efc4" # Verde
    return "#0984e3" # Azul

def actualizar_interfaz(modelo):
    """
    Limpia y redibuja todos los componentes visuales.
    Usamos contenedores Output para asegurar que la actualización es in-place.
    """
    with dashboard_out:
        clear_output(wait=True)
        if modelo is None: return
        
        # Creamos una figura con dos paneles: Mapa y Distribución
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
        agentes = modelo.agents if hasattr(modelo, "agents") else modelo.schedule.agents
        
        # 1. Mapa Social Subjetivo
        x = [a.pos[0] for a in agentes]
        y = [a.pos[1] for a in agentes]
        c = [get_agent_color(a.happiness) for a in agentes]
        ax1.scatter(x, y, c=c, s=30, edgecolors='black', linewidth=0.3)
        ax1.set_title(f"Mapa Social | Paso Nº: {modelo.steps_count}", weight='bold')
        ax1.axis('off')
        
        # 2. Histograma de Felicidad
        h_vals = [a.happiness for a in agentes]
        ax2.hist(h_vals, bins=np.arange(0, 6, 0.5), color='#74b9ff', edgecolor='black', alpha=0.8)
        ax2.set_title("Distribución de Felicidad Real-Time", weight='bold')
        ax2.set_xlabel("Escala (0=Triste, 5=Pleno)")
        
        plt.tight_layout()
        plt.show()

    with stats_out:
        clear_output(wait=True)
        if modelo is None: return
        
        # Generar resumen de bienestar por nivel de renta decil-CIS
        stats = []
        for i in range(1, 12):
            h_cat = [a.happiness for a in agentes if a.income_code == i]
            if h_cat: 
                stats.append({
                    "Escala Ingresos": i, 
                    "Renta Est.": f"{MAPA_EQUIV[i]}€", 
                    "Felicidad Media": round(np.mean(h_cat), 3)
                })
        
        print("\n → ANÁLISIS DE BIENESTAR DETALLADO")
        display(pd.DataFrame(stats))

# Widgets de control
w_red = widgets.Dropdown(options=['model_FB.xlsx', 'model_IG.xlsx', 'model_X.xlsx'], description='Dataset:')
w_smi = widgets.IntSlider(value=1050, min=0, max=5000, step=50, description='SMI (€):', layout=widgets.Layout(width='400px'))
btn_r = widgets.Button(description="RESETEAR MUNDO", button_style='warning', layout=widgets.Layout(width='49%', height='40px'))
btn_s = widgets.Button(description="SIGUIENTE PASO >>", button_style='primary', layout=widgets.Layout(width='49%', height='40px'))

m = None

def handle_reset(b):
    global m
    m = SocialEvolutionModel(300, 30, 30, w_red.value, min_wage=w_smi.value)
    actualizar_interfaz(m)

def handle_step(b):
    global m
    if m is None: handle_reset(None)
    else: 
        m.step()
        actualizar_interfaz(m)

btn_r.on_click(handle_reset)
btn_s.on_click(handle_step)

# Maquetación visual del Dashboard
display(widgets.VBox([
    widgets.HTML("<h4><b>Configuración de la Simulación</b></h4>"),
    widgets.HBox([w_red, w_smi]), 
    widgets.HBox([btn_r, btn_s]), 
    dashboard_out,
    stats_out
]))
print("Panel cargado. Elige tus parámetros y pulsa Resetear para iniciar.")